1(a) Apply one-hot encoding to City and Payment_Method using Pandas or Scikit-learn. Display the transformed DataFrame, verify that the number of rows remains unchanged, and programmatically identify the number of new columns created.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer_ID": range(101, 113),
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Bhopal",
        "Delhi", "Pune", "Mumbai", "Pune",
        "Bhopal", "Delhi", "Pune", "Mumbai"
    ],
    "Payment_Method": [
        "UPI", "Cash", "Card", "UPI",
        "Card", "Cash", "UPI", "Card",
        "Cash", "UPI", "Card", "Cash"
    ],
    "Purchase_Amount": [
        500, 1200, 800, 1500, 700, 900,
        1800, 1100, 600, 1300, 950, 2000
    ]
})

print("Original Data:")
print(df)

df_encoded = pd.get_dummies(
    df,
    columns=["City", "Payment_Method"],
    dtype=int
)

print("\nEncoded Data:")
print(df_encoded)

print("\nOriginal Rows:", df.shape[0])
print("Encoded Rows:", df_encoded.shape[0])

new_columns = [
    col for col in df_encoded.columns
    if col not in df.columns
]

print("New Columns:", new_columns)
print("Number of New Columns:", len(new_columns))


Original Data:
    Customer_ID    City Payment_Method  Purchase_Amount
0           101  Bhopal            UPI              500
1           102   Delhi           Cash             1200
2           103  Mumbai           Card              800
3           104  Bhopal            UPI             1500
4           105   Delhi           Card              700
5           106    Pune           Cash              900
6           107  Mumbai            UPI             1800
7           108    Pune           Card             1100
8           109  Bhopal           Cash              600
9           110   Delhi            UPI             1300
10          111    Pune           Card              950
11          112  Mumbai           Cash             2000

Encoded Data:
    Customer_ID  Purchase_Amount  City_Bhopal  City_Delhi  City_Mumbai  \
0           101              500            1           0            0   
1           102             1200            0           1            0   
2           103     

1(b) Compare the original and encoded DataFrame dimensions, calculate the memory usage of both, and explain why Customer_ID should generally not be one-hot encoded as an ordinary categorical feature.

In [2]:
print("Original Shape:", df.shape)
print("Encoded Shape:", df_encoded.shape)

original_memory = df.memory_usage(deep=True).sum()
encoded_memory = df_encoded.memory_usage(deep=True).sum()

print("\nOriginal Memory:", original_memory, "bytes")
print("Encoded Memory:", encoded_memory, "bytes")

print("\nOriginal Memory in KB:",
      round(original_memory / 1024, 2), "KB")

print("Encoded Memory in KB:",
      round(encoded_memory / 1024, 2), "KB")

if encoded_memory > original_memory:
    print("\nEncoded DataFrame uses more memory.")
elif encoded_memory < original_memory:
    print("\nEncoded DataFrame uses less memory.")
else:
    print("\nBoth use the same memory.")

Original Shape: (12, 4)
Encoded Shape: (12, 9)

Original Memory: 2055 bytes
Encoded Memory: 1060 bytes

Original Memory in KB: 2.01 KB
Encoded Memory in KB: 1.04 KB

Encoded DataFrame uses less memory.


2(a) Build a Scikit-learn preprocessing pipeline that applies one-hot encoding to City and Browser while passing Age through unchanged. Verify that the transformed feature matrix contains only numerical values and report the resulting feature names and dimensions.

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.DataFrame({
    "Age": [22, 35, 28, 45, 31, 24, 40, 29, 38, 26, 33, 42],
    "City": ["Bhopal", "Delhi", "Mumbai", "Pune",
             "Delhi", "Bhopal", "Mumbai", "Pune",
             "Delhi", "Mumbai", "Bhopal", "Pune"],
    "Browser": [
        "Chrome", "Firefox", "Safari", "Chrome",
        "Safari", "Firefox", "Chrome", "Safari",
        "Firefox", "Chrome", "Safari", "Firefox"],
    "Purchase_Amount": [600, 1500, 900, 2200, 1300, 750,
                        2000, 1100, 1800, 850, 1400, 2100]})

X = df[["Age", "City", "Browser"]]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         ["City", "Browser"])
    ],
    remainder="passthrough"
)

X_transformed = preprocessor.fit_transform(X)

X_array = (
    X_transformed.toarray()
    if hasattr(X_transformed, "toarray")
    else X_transformed
)

feature_names = preprocessor.get_feature_names_out()

X_encoded = pd.DataFrame(
    X_array,
    columns=feature_names
)

print("Transformed Data:")
print(X_encoded)

print("\nFeature Names:")
print(list(feature_names))

print("\nDimensions:", X_encoded.shape)

print("All values are numeric:",
      np.issubdtype(X_encoded.to_numpy().dtype, np.number))


Transformed Data:
    categorical__City_Bhopal  categorical__City_Delhi  \
0                        1.0                      0.0   
1                        0.0                      1.0   
2                        0.0                      0.0   
3                        0.0                      0.0   
4                        0.0                      1.0   
5                        1.0                      0.0   
6                        0.0                      0.0   
7                        0.0                      0.0   
8                        0.0                      1.0   
9                        0.0                      0.0   
10                       1.0                      0.0   
11                       0.0                      0.0   

    categorical__City_Mumbai  categorical__City_Pune  \
0                        0.0                     0.0   
1                        0.0                     0.0   
2                        1.0                     0.0   
3               

2(b) Split the data into training and testing sets, fit a regression model through the pipeline, and evaluate it using mean absolute error (MAE). Explain why preprocessing should be fitted on training data and how handle_unknown="ignore" helps when the test data contains an unseen category.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.DataFrame({
    "Age": [22, 35, 28, 45, 31, 24, 40, 29, 38, 26, 33, 42],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Pune",
        "Delhi", "Bhopal", "Mumbai", "Pune",
        "Delhi", "Mumbai", "Bhopal", "Pune"
    ],
    "Browser": [
        "Chrome", "Firefox", "Safari", "Chrome",
        "Safari", "Firefox", "Chrome", "Safari",
        "Firefox", "Chrome", "Safari", "Firefox"
    ],
    "Purchase_Amount": [
        600, 1500, 900, 2200, 1300, 750,
        2000, 1100, 1800, 850, 1400, 2100
    ]
})

X = df[["Age", "City", "Browser"]]
y = df["Purchase_Amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         ["City", "Browser"])
    ],
    remainder="passthrough"
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Actual Purchase Amounts:")
print(y_test.to_list())

print("\nPredicted Purchase Amounts:")
print(y_pred.round(2))

mae = mean_absolute_error(y_test, y_pred)
print("\nMean Absolute Error (MAE):", round(mae, 2))


Actual Purchase Amounts:
[1400, 850, 600]

Predicted Purchase Amounts:
[1524.42  769.77  570.93]

Mean Absolute Error (MAE): 77.91


Q3. High Cardinality and Memory Consumption
An online platform stores product and user information. Some categorical columns contain many unique values.

import pandas as pd

df = pd.DataFrame({
    "Product_Category": [
        "Electronics", "Clothing", "Books", "Electronics",
        "Food", "Books", "Clothing", "Food"
    ],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Pune",
        "Indore", "Delhi", "Bhopal", "Jaipur"
    ],
    "User_ID": [
        "U101", "U102", "U103", "U104",
        "U105", "U106", "U107", "U108"
    ],
    "Purchase_Amount": [
        1200, 800, 400, 2500, 600, 350, 900, 700
    ]
})

(a) Apply One-Hot Encoding and count unique categories
High Cardinality ka matlab hai kisi categorical column mein bahut saari unique values hona. Jaise User_ID mein har user ki alag ID hai.

In [5]:
import pandas as pd

df = pd.DataFrame({
    "Product_Category": [
        "Electronics", "Clothing", "Books", "Electronics",
        "Food", "Books", "Clothing", "Food"
    ],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Pune",
        "Indore", "Delhi", "Bhopal", "Jaipur"
    ],
    "User_ID": [
        "U101", "U102", "U103", "U104",
        "U105", "U106", "U107", "U108"
    ],
    "Purchase_Amount": [
        1200, 800, 400, 2500, 600, 350, 900, 700
    ]
})

# Count unique values
print("Unique values in each column:")
print(df.nunique())

# One-Hot Encode all categorical columns
df_encoded = pd.get_dummies(
    df,
    columns=["Product_Category", "City", "User_ID"],
    dtype=int
)

print("\nEncoded DataFrame:")
print(df_encoded)

print("\nOriginal Shape:", df.shape)
print("Encoded Shape:", df_encoded.shape)
print("New columns created:",
      df_encoded.shape[1] - df.shape[1])


Unique values in each column:
Product_Category    4
City                6
User_ID             8
Purchase_Amount     8
dtype: int64

Encoded DataFrame:
   Purchase_Amount  Product_Category_Books  Product_Category_Clothing  \
0             1200                       0                          0   
1              800                       0                          1   
2              400                       1                          0   
3             2500                       0                          0   
4              600                       0                          0   
5              350                       1                          0   
6              900                       0                          1   
7              700                       0                          0   

   Product_Category_Electronics  Product_Category_Food  City_Bhopal  \
0                             1                      0            1   
1                             0                  